In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

df = pd.read_csv("earthquake_engineered.csv")
feature_cols = ['Latitude', 'Longitude', 'Depth', 'Year', 'Month', 'DayOfWeek', 'RollingMag_30D']
X = df[feature_cols].values
y = df['MagCategory'].astype(str).values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [2]:
# class_weight='balanced' matters here - Major earthquakes are rare (~3% of rows),
# so an unweighted model would just predict 'Minor' most of the time and still look accurate.
models = {
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced'),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=200, random_state=42),
    "Extra Trees": ExtraTreesClassifier(n_estimators=200, random_state=42, class_weight='balanced'),
    "SVC": SVC(class_weight='balanced', random_state=42),
}

In [3]:
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results[name] = {
        "Accuracy": accuracy_score(y_test, pred),
        "Macro Precision": precision_score(y_test, pred, average='macro', zero_division=0),
        "Macro Recall": recall_score(y_test, pred, average='macro', zero_division=0),
        "Macro F1": f1_score(y_test, pred, average='macro', zero_division=0),
    }
    print(f"--- {name} ---")
    print(classification_report(y_test, pred, zero_division=0))

--- Random Forest ---
              precision    recall  f1-score   support

       Major       0.00      0.00      0.00       148
       Minor       0.70      0.94      0.80      3186
    Moderate       0.44      0.12      0.19      1312

    accuracy                           0.68      4646
   macro avg       0.38      0.35      0.33      4646
weighted avg       0.61      0.68      0.61      4646



--- Gradient Boosting ---
              precision    recall  f1-score   support

       Major       0.29      0.01      0.03       148
       Minor       0.70      0.97      0.81      3186
    Moderate       0.54      0.11      0.18      1312

    accuracy                           0.69      4646
   macro avg       0.51      0.36      0.34      4646
weighted avg       0.64      0.69      0.61      4646



--- Extra Trees ---
              precision    recall  f1-score   support

       Major       0.00      0.00      0.00       148
       Minor       0.70      0.91      0.79      3186
    Moderate       0.38      0.13      0.19      1312

    accuracy                           0.66      4646
   macro avg       0.36      0.35      0.33      4646
weighted avg       0.59      0.66      0.60      4646



--- SVC ---
              precision    recall  f1-score   support

       Major       0.05      0.44      0.08       148
       Minor       0.75      0.51      0.61      3186
    Moderate       0.31      0.25      0.27      1312

    accuracy                           0.44      4646
   macro avg       0.37      0.40      0.32      4646
weighted avg       0.60      0.44      0.50      4646



In [4]:
results_df = pd.DataFrame(results).T
results_df.to_csv("classification_results.csv")
print(results_df)

                   Accuracy  Macro Precision  Macro Recall  Macro F1
Random Forest      0.679724         0.380564      0.354613  0.332132
Gradient Boosting  0.694146         0.509087      0.363378  0.341314
Extra Trees        0.663366         0.358549      0.347558  0.327981
SVC                0.435213         0.367489      0.399266  0.321722
